# Official end-to-end token accounting
This replaces the former synthetic end-to-end token workload with official SWE-Bench Lite and Terminal-Bench records.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'
PAPER = ROOT / 'paper' / 'img'
PAPER.mkdir(parents=True, exist_ok=True)
frames = []
for backend in ('deepseek_harness', 'codex', 'official'):
    path = RESULTS / backend / 'official_token_summary.csv'
    if path.exists(): frames.append(pd.read_csv(path))
df = pd.concat(frames, ignore_index=True).drop_duplicates() if frames else pd.DataFrame()
if not df.empty:
    view = df.groupby(['suite', 'scale', 'mode'], as_index=False)[['prompt_tokens_mean', 'completion_tokens_mean', 'total_tokens_mean']].mean()
    fig, ax = plt.subplots(figsize=(6.8, 3.0), dpi=300)
    for (suite, mode), group in view.groupby(['suite', 'mode']):
        group = group.sort_values('scale')
        ax.plot(group['scale'], group['prompt_tokens_mean'], marker='o', linewidth=1.0, label=f'{suite}:{mode}:prompt')
        ax.plot(group['scale'], group['completion_tokens_mean'], marker='x', linewidth=0.9, linestyle='--', label=f'{suite}:{mode}:completion')
    ax.set_xlabel('Official task scale')
    ax.set_ylabel('Tokens (mean)')
    ax.legend(fontsize=5.5, ncol=2, frameon=True)
    fig.tight_layout()
    fig.savefig(FIGDIR / 'FIG-Token-End-to-End.pdf', bbox_inches='tight')
    fig.savefig(PAPER / 'FIG-Token-End-to-End.pdf', bbox_inches='tight')
else:
    print('No official summary found; run the official benchmark first.')
# Compatibility source name: token_end_to_end.csv; official_token_summary.csv is authoritative.
